# 14 · Swing 1º turno 2022 → 1º turno 2026

**Comparação principal**: 1º turno 2022 → 1º turno 2026, % sobre **válidos de cada ano**, por município pareado (chave `uf` + `cd_mun_tse`). Swing = p.p. de 2026 − p.p. de 2022.

**Identidade (obrigatória)**:
- **PT**: Lula (nº 13) nos dois anos, mesma pessoa.
- **direita (nº 22)**: **Jair Bolsonaro/PL em 2022** → **Flávio Bolsonaro/PL em 2026**. O swing do nº 22 mede a sigla/numeração, **não** a mesma pessoa.

**Sem par** (fora de qualquer swing e de qualquer agregado, listados explicitamente): município novo de MT (Boa Esperança do Norte), postos consulares novos e extintos, e postos com **zero votos válidos** em um dos anos (% indefinido, 0/0).

**Agregado ponderado** = soma de votos ÷ soma de válidos, nos dois anos, com a mesma base de municípios. A média simples do swing municipal aparece só para comparação.

**Exterior** não entra nos agregados Brasil.

In [ ]:
import sys, pathlib

sys.path.insert(0, str(pathlib.Path("..") / "src"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
from eleicao.analise import composicao as cp, base, carga, graficos as g
from eleicao.analise import (
    participacao as pa,
    concentracao as cc,
    swing as sw,
    perfil,
    espacial as esp,
    bolsoes as bo,
    mapa_mente as mm,
)

In [ ]:
dados = cp.swing_pt_pl()
pt, pl, sem = dados["pt"], dados["pl"], dados["sem_par"]
print("pareados (com exterior):", len(pl), "| sem par ou excluídos:", len(sem))
print(sem.to_string(index=False))

## Agregados ponderados por região (Brasil, sem exterior)

In [ ]:
pl_br, pt_br = pl[pl.uf != "zz"], pt[pt.uf != "zz"]
resumo = pd.concat(
    [
        sw.agregado_ponderado(pt_br, "regiao").assign(candidato="PT (13)"),
        sw.agregado_ponderado(pl_br, "regiao").assign(candidato="PL (22)"),
    ]
)
media = pd.concat(
    [
        sw.media_simples_swing(pt_br, "regiao").assign(candidato="PT (13)"),
        sw.media_simples_swing(pl_br, "regiao").assign(candidato="PL (22)"),
    ]
)
print(
    resumo[["candidato", "regiao", "pct_2022_agregado", "pct_2026_agregado", "swing_agregado_pp"]]
    .round(2)
    .to_string(index=False)
)
print(media.round(2).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
reg = cp.ponderar_regioes(resumo)
regs = list(dict.fromkeys(reg.regiao))
x = np.arange(len(regs))
w = 0.38
for i, (cnd, nr) in enumerate([("PT (13)", 13), ("PL (22)", 22)]):
    vals = cp.ponderar_regioes(resumo[resumo.candidato == cnd]).swing_agregado_pp.to_numpy()
    ax.bar(x + (i - 0.5) * w, vals, w, color=g.cor_candidato(nr), label=cnd)
    for xi, v in zip(x + (i - 0.5) * w, vals):
        ax.text(xi, v + (0.15 if v >= 0 else -0.5), f"{v:+.1f}", ha="center", fontsize=8)
ax.axhline(0, color="#333", lw=0.8)
ax.set_xticks(x, regs)
ax.set_ylabel("swing agregado (p.p. de válidos)")
ax.set_title(
    "Swing 1T 2022→2026 por região (ponderado). Nº 22: Bolsonaro (2022) → Flávio (2026)",
    loc="left",
    fontsize=10,
)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
g.salvar(fig, "swing_agregado_regiao")
plt.show()

## Teste de uniformidade: o swing difere entre regiões mais do que dentro delas?

In [ ]:
rows = []
for nome, tab_ in (("PT (13)", pt_br), ("PL (22)", pl_br)):
    r = sw.teste_uniformidade(tab_, "regiao")
    rows.append(
        {
            "candidato": nome,
            **{k: (round(v, 6) if isinstance(v, float) else v) for k, v in r.items()},
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

Interpretação: com n ≈ 5.570, qualquer diferença pequena vira “significativa”. O que importa é o **eta²** (fração da variância municipal do swing explicada pela região). Ver `docs/ANALISES.md`.

## Distribuição municipal do swing

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.hist(pt_br.swing_pp, bins=80, alpha=0.6, color=g.cor_candidato(13), label="PT (13)")
ax.hist(pl_br.swing_pp, bins=80, alpha=0.6, color=g.cor_candidato(22), label="PL (22)")
ax.axvline(0, color="#333", lw=0.8)
ax.set_xlabel("swing municipal (p.p. de válidos, 2026 − 2022)")
ax.set_ylabel("municípios")
ax.set_title("Swing municipal 1T 2022→2026 (municípios BR pareados)", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
g.salvar(fig, "swing_histograma")
plt.show()
print(
    "mediana PT:",
    round(pt_br.swing_pp.median(), 2),
    "| mediana PL:",
    round(pl_br.swing_pp.median(), 2),
)

## Mapa do swing do PL (municípios sem par ficam sem cor e estão listados acima)

In [ ]:
b = cp.base_2026()
m = cp.malha_swing(b)
fig = g.mapa_valores(
    m,
    "swing_pp",
    "Swing do nº 22 (PL): Bolsonaro 2022 → Flávio 2026, p.p. de válidos",
    cmap="RdBu_r",
    centro=0.0,
    rotulo_cbar="p.p.",
)
g.salvar(fig, "swing_mapa_pl")
plt.show()

## Lente secundária: PL 2026 (1º turno) × teto de Bolsonaro no 2º turno 2022

**Teto** = % de válidos de Bolsonaro (nº 22) no **2º turno 2022**, no próprio município. Diferença positiva = PL 2026 (1T) **acima** desse teto.

**Ressalva principal**: 1º e 2º turno **não são diretamente comparáveis**. O 2º turno é binário, com outro comparecimento e dinâmica de rejeição; a lente mostra a posição relativa a um número de referência, não uma projeção.

In [ ]:
lente = cp.lente_pl_teto()
ag_lente = sw.lente_regiao(lente)
print(ag_lente.round(2).to_string(index=False))
print(sw.contagem_acima_do_teto(lente).round(1).to_string(index=False))

In [ ]:
ag = cp.ponderar_regioes(ag_lente)
fig, ax = plt.subplots(figsize=(9, 4.6))
x = np.arange(len(ag))
w = 0.38
ax.bar(
    x - w / 2,
    ag.teto_2t_2022_agregado_pp,
    w,
    color="#999999",
    label="teto: Bolsonaro 2º turno 2022",
)
ax.bar(
    x + w / 2,
    ag.pl_2026_1t_agregado_pp,
    w,
    color=g.cor_candidato(22),
    label="PL 2026 (nº 22, 1º turno)",
)
ax.set_xticks(x, ag.regiao)
ax.set_ylabel("% dos válidos (ponderado)")
ax.set_title(
    "PL 2026 (1T) × teto Bolsonaro 2T 2022: NÃO comparáveis diretamente", loc="left", fontsize=10
)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
g.salvar(fig, "lente_teto_regiao")
plt.show()